In [7]:
import os
import json
import warnings
import numpy as np
import xarray as xr
import proplot as pplt
warnings.filterwarnings('ignore')
pplt.rc.update({
    'savefig.dpi':900,
    'savefig.bbox':'tight',
    'savefig.pad_inches':0.02,
    'tick.minor':False,
    'font.size':9,
    'label.size':9,
    'tick.labelsize':9,
    'title.size':9,
    'abc.size':9,
    'legend.fontsize':9,
    'suptitle.size':9,
    'leftlabelsize':9,
    'toplabelsize':9,
    'leftlabel.weight':'normal',
    'toplabel.weight':'normal',
    'reso':'xx-hi'})

In [8]:
with open('../scripts/configs.json','r',encoding='utf-8') as f:
    CONFIGS = json.load(f)
SPLITSDIR  = CONFIGS['filepaths']['splits']
INTERIMDIR = CONFIGS['filepaths']['interim']
LATRANGE   = CONFIGS['domain']['latrange']
LONRANGE   = CONFIGS['domain']['lonrange']

In [9]:
def mean(da):
    if 'time' not in da.dims:
        return da
    return da.mean('time',skipna=True)

In [16]:
lf,shf = [],[]
for split in ['train','valid']:
    with xr.open_dataset(os.path.join(SPLITSDIR,f'norm_{split}.h5'),engine='h5netcdf') as ds:
        lf.append(ds['lf'].load())
        shf.append(ds['shf'].load())
lf  = lf[0] if 'time' not in lf[0].dims else xr.concat(lf,dim='time')
shf = xr.concat(shf,dim='time')
lfmean  = mean(lf)
shfmean = mean(shf)
lf      = lf.broadcast_like(shf) if 'time' not in lf.dims else lf
maxmean = mean(xr.apply_ufunc(np.maximum,lf,shf))

with xr.open_dataset(os.path.join(INTERIMDIR,'sst.nc')) as ds:
    sst = ds['sst'].load()
sstmean = mean(sst)
sstanom = sstmean-sstmean.mean(skipna=True)

In [ ]:
climlim = float(max(np.nanmax(np.abs(lfmean.values)),np.nanmax(np.abs(shfmean.values)),np.nanmax(np.abs(maxmean.values))))
sstlim  = float(np.nanmax(np.abs(sstanom.values)))

fig,axs = pplt.subplots(nrows=2,ncols=2,proj='cyl',figwidth=6.5,share=True)
axs.format(coast=True,borders=False,latlim=LATRANGE,lonlim=LONRANGE,latlines=[10,15,20],lonlines=5,grid=False)
axs[-1,:].format(lonlabels='b')
axs[:,0].format(latlabels='l')
axs[0].format(title=r'$\widetilde{LF}$')
axs[1].format(title=r'$\widetilde{SHF}$')
axs[2].format(title=r'$\max(\widetilde{LF},\widetilde{SHF})$')
axs[3].format(title='SST Anomalies')

kw = dict(cmap='ColdHot',vmin=-climlim,vmax=climlim,levels=21)
m1 = axs[0].pcolormesh(lfmean.lon,lfmean.lat,lfmean,**kw)
axs[1].pcolormesh(shfmean.lon,shfmean.lat,shfmean,**kw)
axs[2].pcolormesh(maxmean.lon,maxmean.lat,maxmean,**kw)
m2 = axs[3].pcolormesh(sstanom.lon,sstanom.lat,sstanom,cmap='ColdHot',vmin=-sstlim,vmax=sstlim,levels=21)

fig.colorbar(m1,ax=[axs[0],axs[1]],loc='b',label='Standard Deviations')
fig.colorbar(m1,ax=axs[2],loc='b',label='Standard Deviations')
fig.colorbar(m2,ax=axs[3],loc='b',label=r'$\Delta$K')
pplt.show()
fig.save('../figs/fig_2.jpg')